# Build table of recommended algorithms, loading in local street networks

This notebook loops through cities, loads and calculates street and bike network properties, and decides based on those properties which BikeNetKit algorithm should be recommended.

Outputs a ranked list of algorithms in a csv file that make sense for each city, like this:

```
slug,gbn,gbn+,lbn,fbn,sb
copenhagen,0,0,0,1,1
losangeles,1,0,0,0,1
budapest,0,1,1,1,1
...
```

Recommend algorithms as follows (to be tweaked):
```
sb: True
gbn: cov_ratio < 0.15 or l_ratio < 0.05
gbn+: (cov_ratio > 0.1 and cov_ratio < 0.6) or (l_ratio > 0.03 and l_ratio < 0.2)
lbn: gcc3 > 0 and gcc1+gcc2+gcc3 < 0.8 and cov_ratio > 0.15 and l_ratio > 0.04
fbn: l_ratio >= 0.25 or ((gcc1 >= 0.25 or gcc1+gcc2+gcc3 >= 0.5) and cov_ratio >= 0.4 and l_ratio >= 0.1)
```

where:
- `cov_ratio` is the buffered bike network's area fraction of the area of the whole buffered street network
- `l_ratio` is the bike network's length fraction of the length of the whole street network
- `gcc1`, `gcc2`, `gcc3` are the fractions of the bike network's lengths in its largest, second, and third largest connected components

## Load packages

In [ ]:
import pyrosm
import csv
import growbikenet as gbn
from growbikenet.functions import *
import matplotlib.pyplot as plt
print(gbn.__version__)

## Parameters

In [ ]:
countries_path = "../countries/"
cities_path = "../cities/"
cityfilename = "cities.csv"
bufferlength = 500
make_plots = False

In [ ]:
def calc_rec_this(cov_ratio, l_ratio, gcc1, gcc2, gcc3):
    rec_this = {"cityid": cityid, "sb": 1, "gbn": 0, "gbn+": 0, "lbn": 0, "fbn": 0}
    if cov_ratio < 0.15 or l_ratio < 0.05:
        rec_this["gbn"] = 1
    if (cov_ratio > 0.1 and cov_ratio < 0.6) or (l_ratio > 0.03 and l_ratio < 0.2):
        rec_this["gbn+"] = 1
    if gcc3 > 0 and gcc1+gcc2+gcc3 < 0.8 and cov_ratio > 0.15 and l_ratio > 0.04:
        rec_this["lbn"] = 1
    if l_ratio >= 0.25 or ((gcc1 >= 0.2 or gcc1+gcc2+gcc3 >= 0.5) and cov_ratio >= 0.4 and l_ratio >= 0.1):
        rec_this["fbn"] = 1
    return rec_this

## Load cities

In [ ]:
with open(cities_path+'meta/'+cityfilename, mode='r') as infile:
    reader = csv.reader(infile, delimiter=";")
    header = next(reader)
    cities = {rows[0]: {header[1]: rows[1], header[2]: rows[2], header[3]: rows[3], header[4]: rows[4]}  for rows in reader}

## Run by importing cities and calculating

In [ ]:
rec_table = []
network_metrics = []
for cityid, city_info in cities.items():
    print(city_info["name_en"], cityid)
    
    network_metrics_this = {"cityid": cityid, "cov_ratio": 0, "l_ratio": 0, "gcc1": 0, "gcc2": 0, "gcc3": 0}

    # LOAD NETWORKS
    
    if os.path.exists(cities_path+"cityexport/street_networks/"+cityid+".gpkg"):
        nodes_car, edges_car, g_undir_car, _ = import_network(cityid+".gpkg", import_path=cities_path+"cityexport/street_networks/")

        if os.path.exists(cities_path+"cityexport/bike_networks/"+cityid+".gpkg"):
            nodes_bike, edges_bike, g_undir_bike, _ = import_network(cityid+".gpkg", import_path=cities_path+"cityexport/bike_networks/")
        
            edges_car_buffer = edges_car.buffer(bufferlength)
            edges_bike_buffer = edges_bike.buffer(bufferlength)

            # Yeah, the next line is a mouthful. It is the components ordered by total link length.
            S = [g_undir_bike.subgraph(c).copy() for c in sorted(nx.connected_components(g_undir_bike), key=lambda c: sum([l[-1] for l in g_undir_bike.subgraph(c).copy().edges.data('length')]), reverse=True)]

            if make_plots:
                fig = plt.figure(figsize=(4,4))
                ax = fig.add_axes([0,0,1,1])
                edges_car_buffer.plot(ax=ax, color="grey")
                edges_car.plot(ax=ax, color="k", linewidth=0.5)
                edges_bike_buffer.plot(ax=ax, color="orange")
                edges_bike.plot(ax=ax, color="r", linewidth=1)
                plt.text(x=0.5, y=1.03, s=city_info["country_en"], fontsize=9, ha="center", transform=fig.transFigure, color="grey")
                plt.text(x=0.5, y=0.98, s=city_info["name_en"], fontsize=12, ha="center", transform=fig.transFigure)

                # gcc1,2,3
                for comp in [0,1,2]:
                    if len(S)>=comp+1:
                        edges_component = ox.graph_to_gdfs(S[comp], nodes=False, edges=True, node_geometry=False, fill_edge_geometry=False)
                        edges_component.to_crs(edges_bike.crs).plot(ax=ax, color="#"+str(99-(2-comp)*22)+"00"+str(10+(2-comp)*22), linewidth=2)

                ax.axis('off')
                fig.savefig(f"../cities/plots/{cityid}.png", dpi=150, bbox_inches='tight')
                plt.close()
        
            # CALCULATE METRICS
            cov_ratio = edges_bike_buffer.union_all().area / edges_car_buffer.union_all().area
            l_ratio = sum(edges_bike.length) / sum(edges_car.length)
            edges_bike_length = sum([l[-1] for l in g_undir_bike.edges.data('length')])
            gcc1 = sum([l[-1] for l in S[0].edges.data('length')]) / edges_bike_length
            if len(S) >= 2:
                gcc2 = sum([l[-1] for l in S[1].edges.data('length')]) / edges_bike_length
            if len(S) >= 3:
                gcc3 = sum([l[-1] for l in S[2].edges.data('length')]) / edges_bike_length
        
        else: # There is no bike infra
            cov_ratio = l_ratio = gcc1 = edges_bike_length = 0
            gcc1 = gcc2 = gcc3 = 0
            
        # print("cov_ratio: " + str(cov_ratio))
        # print("l_ratio: " + str(l_ratio))
        # print("gcc1: " + str(gcc1))
        # print("gcc2: " + str(gcc2))
        # print("gcc3: " + str(gcc3))
    
        # RECOMMEND ALGORITHMS
        rec_this = calc_rec_this(cov_ratio, l_ratio, gcc1, gcc2, gcc3)

        network_metrics_this["cov_ratio"] = f"{cov_ratio:.2f}" 
        network_metrics_this["l_ratio"] = f"{l_ratio:.2f}" 
        network_metrics_this["gcc1"] = f"{gcc1:.2f}" 
        network_metrics_this["gcc2"] = f"{gcc2:.2f}" 
        network_metrics_this["gcc3"] = f"{gcc3:.2f}" 

        rec_table.append(rec_this)
        network_metrics.append(network_metrics_this)

### Export network metrics

In [ ]:
with open(cities_path+'meta/network_metrics.csv', 'w', newline='') as csvfile:
    fieldnames = ['cityid', 'cov_ratio', 'l_ratio', 'gcc1', 'gcc2', 'gcc3']
    writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(network_metrics)

## Run by importing already calculated metrics

If cells above were run already, then only everything below needs to be run to re-build the rec-table:

In [ ]:
with open(cities_path+'meta/network_metrics.csv', mode='r') as infile:
    reader = csv.reader(infile, delimiter=",")
    header = next(reader)
    metrics = {rows[0]: {header[0]: rows[0], header[1]: float(rows[1]), header[2]: float(rows[2]), header[3]: float(rows[3]), header[4]: float(rows[4]), header[5]: float(rows[5])} for rows in reader}

rec_table = []
for cityid, metrics_this in metrics.items():
    rec_this = calc_rec_this(metrics_this["cov_ratio"], metrics_this["l_ratio"], metrics_this["gcc1"], metrics_this["gcc2"], metrics_this["gcc3"])
    rec_table.append(rec_this)

## Export data

In [ ]:
with open(cities_path+'meta/rec_table.csv', 'w', newline='') as csvfile:
    fieldnames = ['cityid', 'sb', 'gbn', 'gbn+', 'lbn', 'fbn']
    writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(rec_table)

In [ ]:
rec_table_sum = [sum([d['sb'] for d in rec_table]),
                 sum([d['gbn'] for d in rec_table]),
                sum([d['gbn+'] for d in rec_table]),
                sum([d['lbn'] for d in rec_table]),
                sum([d['fbn'] for d in rec_table]),
                sum([max(d['gbn'],d['gbn+']) for d in rec_table]),
]
print('  sb  gbn gbn+  lbn  fbn  gbn/gbn+\n',rec_table_sum)